# HireShield-AI — Certificate Forgery Detector (MobileNetV3-Small)

Fine-tunes a lightweight CNN to separate **genuine** internship certificates from
**forged** ones, then exports it to ONNX so the backend only needs `onnxruntime`
(no PyTorch in production).

**Dataset:** [Internship Certificates — godzilla04](https://www.kaggle.com/datasets/godzilla04/internship-certificates)
— CC0 Public Domain, 287 images (165 real / 122 fake).

## How to run

### Kaggle (recommended — free GPU)

1. **File → Import Notebook** and upload this file.
2. Right sidebar → **Add Input → Datasets** → search `internship-certificates`
   (owner `godzilla04`) → **Add**. It mounts at `/kaggle/input/internship-certificates`.
3. **Settings → Accelerator → GPU T4 x2** (or P100).
4. **Settings → Internet → On.** *Required*: the ImageNet weights for transfer
   learning are downloaded at runtime. With Internet off the notebook still runs but
   trains from scratch and prints a loud warning — accuracy drops sharply on ~200
   training images.
5. **Run All** — training takes roughly 5–10 minutes.

> Kaggle's working directory is `/kaggle/working`, so `certificate_cnn.onnx`,
> `certificate_cnn.json` and `certificate_metrics.json` appear directly in the
> right-hand **Output** panel when the run finishes.

See `KAGGLE_RUNBOOK.md` in this folder for the full copy-out checklist.

### Colab
1. Upload this notebook; **Runtime → Change runtime type → T4 GPU**.
2. Run the first code cell. If the data is not mounted it downloads via the Kaggle API
   (needs a Kaggle token), otherwise upload the dataset zip to `data/raw/certificates/`.

## Outputs

| Artifact | Purpose |
|---|---|
| `certificate_cnn.onnx` | Model shipped to the backend (`ml/artifacts/`) |
| `certificate_cnn.json` | Preprocessing + threshold + class sidecar |
| `certificate_metrics.json` | Accuracy, F1, ROC-AUC, confusion matrix, history |
| `confusion_matrix.png`, `roc_curve.png` | Plots for `ml/evaluation/plots/` |

> **Honest limitation (measured, not assumed).** With only 287 images this model learns
> *this dataset*, not forgery in general. Worse, the Kaggle "fake" files are far smaller
> than the real ones (median **256x197 / 8 KB** vs **1190x1683 / 278 KB**), so width,
> height and file size alone separate the classes with **ROC-AUC 0.97** — verified
> below with a logistic-regression baseline. The notebook therefore rescales every
> image to a common size before training and reports the *residual* leakage, so the
> reported score cannot be an artifact of resolution. Even so, inside HireShield-AI
> this model is a **secondary signal only** — it may add supporting evidence but must
> never flip a verdict on its own.


In [ ]:
# --- Environment check / light installs -------------------------------------
import importlib
import subprocess
import sys


def _ensure(module, package=None):
    """Install `package` only when `module` is missing (Kaggle already has torch)."""
    if importlib.util.find_spec(module) is None and package:
        print(f"installing {package} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=False)


_ensure("onnx", "onnx")
_ensure("onnxruntime", "onnxruntime")
_ensure("seaborn", "seaborn")

import torch
import torchvision

print("torch      :", torch.__version__)
print("torchvision:", torchvision.__version__)
print("device     :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (fallback)")


In [ ]:
# --- Config -----------------------------------------------------------------
import hashlib
import io
import json
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMG_SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]
CLASSES = ["real", "fake"]        # index 0 = genuine, index 1 = forged
EPOCHS = 25
BATCH_SIZE = 16
LR = 3e-4
KAGGLE_SLUG = "godzilla04/internship-certificates"

# The Kaggle "fake" images are much smaller than the real ones (median 256x197/8 KB
# vs 1190x1683/278 KB), so file size alone separates the classes with ~0.97 ROC-AUC.
# `NORMALISE_IMAGES` rescales every image to a common short side first, which removes
# the resolution cue; the residual leakage is measured again in the next cell.
NORMALISE_IMAGES = True
NORMALISE_MAX_SIDE = 384          # target for min(width, height) after rescaling
NORMALISE_ORIENTATION = True      # rotate landscape scans upright (see ablation)
NORMALISED_DIR = Path("certificates_normalised")


def find_dataset_root():
    """Locate the class folders on Kaggle, Colab or a local checkout."""
    candidates = [
        Path("/kaggle/input/internship-certificates"),
        Path("/kaggle/input"),
        Path("ml/data/raw/certificates"),
        Path("../data/raw/certificates"),
        Path("data/raw/certificates"),
    ]
    for candidate in candidates:
        if not candidate.is_dir():
            continue
        for directory in [candidate] + [d for d in candidate.iterdir() if d.is_dir()]:
            name = directory.name.lower()
            if "real" in name or "fake" in name:
                return candidate
    return None


ROOT = find_dataset_root()
if ROOT is None:
    print("Dataset not found in the usual locations -- trying the Kaggle API ...")
    os.makedirs("data/raw/certificates", exist_ok=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=False)
    subprocess.run(
        ["kaggle", "datasets", "download", "-d", KAGGLE_SLUG,
         "-p", "data/raw/certificates", "--unzip"],
        check=False,
    )
    ROOT = find_dataset_root()

if ROOT is None:
    raise FileNotFoundError(
        "Could not locate the dataset. On Kaggle use 'Add Input', or run "
        "ml/training/download_certificates.py locally first."
    )

print("dataset root:", ROOT)
print("device      :", DEVICE)


In [ ]:
# --- Collect image paths and labels -----------------------------------------
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}


def label_from_path(path):
    """Label from the folder names ("real"/"fake" anywhere in the path)."""
    for part in path.parts:
        lowered = part.lower()
        if "fake" in lowered or "forged" in lowered:
            return 1
        if "real" in lowered or "genuine" in lowered:
            return 0
    return None


records = []
for path in sorted(ROOT.rglob("*")):
    if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES:
        label = label_from_path(path)
        if label is not None:
            records.append({"path": str(path), "label": label})

assert records, "No labelled images found -- class folders must contain 'real'/'fake'."

labels = pd.Series([r["label"] for r in records]).map({0: "real", 1: "fake"})
print("images found:", len(records))
print(labels.value_counts())


In [ ]:
# --- Integrity, duplicate and size audit ------------------------------------
def md5_of(path, chunk=1 << 20):
    digest = hashlib.md5()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(chunk), b""):
            digest.update(block)
    return digest.hexdigest()


rows, seen, unreadable = [], set(), 0
for record in records:
    try:
        with Image.open(record["path"]) as image:
            image.verify()
        with Image.open(record["path"]) as image:
            width, height = image.size
            mode, fmt = image.mode, image.format
    except Exception:
        unreadable += 1
        continue
    digest = md5_of(record["path"])
    if digest in seen:
        continue
    seen.add(digest)
    rows.append({
        **record,
        "md5": digest,
        "width": width,
        "height": height,
        "mode": mode,
        "format": fmt,
        "bytes": os.path.getsize(record["path"]),
    })

df = pd.DataFrame(rows)
dropped_dupes = len(records) - unreadable - len(df)
print(f"unreadable dropped: {unreadable} | duplicate dropped: {dropped_dupes}")
print(f"usable images     : {len(df)}")
print(df.groupby("label")[["width", "height", "bytes"]].median().round(0))


In [ ]:
# --- Shortcut check: can file metadata alone separate the classes? -----------
# Kaggle "fake" sets are often produced at a different resolution or file size than
# the real ones. If metadata alone predicts the label, the CNN may be learning
# "which pipeline made this file" instead of "is this forged" -- which means the
# model's score is weak evidence and the report must say so.
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

metadata_pipe = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced"),
)
metadata_auc = float(
    cross_val_score(
        metadata_pipe,
        df[["width", "height", "bytes"]].values,
        df["label"].values,
        cv=5,
        scoring="roc_auc",
    ).mean()
)
print(f"metadata-only ROC-AUC (5-fold): {metadata_auc:.3f}")
if metadata_auc > 0.90:
    print(">>> HIGH shortcut risk: resolution/file size leaks the label.")
    print(">>> Keep the CNN as a WEAK signal and do not let it change a risk band.")
else:
    print(">>> Acceptable: the CNN has to look at image content to separate the classes.")


In [ ]:
# --- Mitigation: normalise size AND orientation, then re-audit ---------------
# Raw resolution leaks the label (0.97 AUC, see the cell above), and so does
# orientation: 87% of genuine certificates are portrait while only 19% of the fakes
# are. Rescaling to a common short side and rotating every landscape scan upright
# removes both cues; whatever leakage survives is the honest "pipeline fingerprint"
# the CNN could still exploit, and the report must say so.
def normalise_dataset(frame, target=384, upright=True, out_dir=NORMALISED_DIR):
    """Rescale to a common short side (and optionally make every scan portrait)."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    records, rotated = [], 0
    for row in frame.to_dict("records"):
        source = Path(row["path"])
        with Image.open(source) as handle:
            image = handle.convert("RGB")
            if upright and image.width > image.height:
                image = image.transpose(Image.ROTATE_90)
                rotated += 1
            scale = target / max(1, min(image.size))
            new_size = (max(1, round(image.width * scale)), max(1, round(image.height * scale)))
            image = image.resize(new_size, Image.LANCZOS)
            destination = out_dir / f"{row['label']}_{source.stem}.jpg"
            image.save(destination, format="JPEG", quality=92)
        records.append({
            "path": str(destination),
            "label": int(row["label"]),
            "width": new_size[0],
            "height": new_size[1],
            "bytes": destination.stat().st_size,
        })
    if upright:
        print(f"rotated {rotated} landscape scans upright")
    return pd.DataFrame(records)


def metadata_auc_of(frame):
    """5-fold ROC-AUC of a logistic regression on width/height/bytes."""
    return float(
        cross_val_score(
            metadata_pipe,
            frame[["width", "height", "bytes"]].values,
            frame["label"].values,
            cv=5,
            scoring="roc_auc",
        ).mean()
    )


if NORMALISE_IMAGES:
    raw_auc = metadata_auc_of(df)
    normalised_df = normalise_dataset(
        df, target=NORMALISE_MAX_SIDE, upright=NORMALISE_ORIENTATION
    )
    normalised_auc = metadata_auc_of(normalised_df)
    print(f"metadata ROC-AUC  before : {raw_auc:.3f}   (raw resolution + orientation)")
    print(f"metadata ROC-AUC  after  : {normalised_auc:.3f}   (size + orientation fixed)")
    print(f"leakage removed          : {raw_auc - normalised_auc:+.3f}")
    if normalised_auc > 0.80:
        print(">>> Residual leakage still high: keep the CNN as a WEAK signal, and do")
        print(">>> not let its score change a risk band or a verdict.")
    else:
        print(">>> Residual leakage near chance: the CNN must read content, not metadata.")
    df = normalised_df
    print()
    print(df.groupby("label")[["width", "height", "bytes"]].median().round(0))
else:
    print("NORMALISE_IMAGES is off -- using the raw files (expect high shortcut risk).")


In [ ]:
# --- Stratified 70 / 15 / 15 split ------------------------------------------
from sklearn.model_selection import train_test_split

train_df, holdout = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=SEED)
val_df, test_df = train_test_split(holdout, test_size=0.50, stratify=holdout["label"], random_state=SEED)

for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    counts = part["label"].value_counts().to_dict()
    print(f"{name:5s} n={len(part):4d}  real={counts.get(0, 0):3d}  fake={counts.get(1, 0):3d}")


In [ ]:
# --- Transforms, dataset and loaders ----------------------------------------
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode


class JPEGRecompress:
    """Randomly re-save through JPEG.

    Forged certificates are usually screenshots or Canva exports, so a "clean PNG"
    must not become a shortcut. Recompressing simulates messaging-app forwarding.
    """

    def __init__(self, qualities=(55, 95), p=0.5):
        self.qualities = qualities
        self.p = p

    def __call__(self, image):
        if random.random() > self.p:
            return image
        buffer = io.BytesIO()
        image.convert("RGB").save(
            buffer, format="JPEG", quality=random.randint(*self.qualities)
        )
        buffer.seek(0)
        return Image.open(buffer).convert("RGB")


train_tf = transforms.Compose([
    JPEGRecompress(p=0.5),
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=InterpolationMode.BICUBIC),
    # No horizontal flip: mirrored text never occurs in a real scan, and teaching the
    # model flip-invariance on 277 images only dilutes the signal. Orientation is
    # already fixed on disk by `normalise_dataset`.
    transforms.RandomApply([transforms.RandomRotation(degrees=7, fill=255)], p=0.5),
    transforms.RandomAffine(degrees=0, translate=(0.03, 0.03), scale=(0.95, 1.05), fill=255),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.15),
    transforms.RandomApply([transforms.GaussianBlur(3, sigma=(0.1, 1.5))], p=0.25),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


class CertificateDataset(Dataset):
    def __init__(self, frame, transform):
        self.paths = frame["path"].tolist()
        self.labels = frame["label"].astype(int).tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image:
            image = image.convert("RGB")
        return self.transform(image), self.labels[index]


loaders = {
    "train": DataLoader(CertificateDataset(train_df, train_tf), batch_size=BATCH_SIZE,
                        shuffle=True, num_workers=2),
    "val": DataLoader(CertificateDataset(val_df, eval_tf), batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=2),
    "test": DataLoader(CertificateDataset(test_df, eval_tf), batch_size=BATCH_SIZE, shuffle=False,
                       num_workers=2),
}
for split, loader in loaders.items():
    print(split, len(loader), "batches")


In [ ]:
# --- Model: ImageNet-pretrained MobileNetV3-Small, 2-way head ----------------
# Downloading the ImageNet weights needs Internet enabled in the Kaggle settings.
# If it is off (or the mirror is unreachable) we fall back to random init so the
# notebook still runs -- accuracy will be much worse, and the cell says so.
from torchvision.models import MobileNet_V3_Small_Weights, mobilenet_v3_small

PRETRAINED = False
try:
    model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1)
    PRETRAINED = True
    print("ImageNet weights loaded -- transfer learning enabled.")
except Exception as error:  # noqa: BLE001 - any download/network failure
    model = mobilenet_v3_small(weights=None)
    print(f"WARNING: could not load ImageNet weights ({type(error).__name__}: {error})")
    print("WARNING: training from scratch. Enable Internet in Kaggle settings and re-run")
    print("WARNING: this cell for much better accuracy on only ~200 training images.")

model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, len(CLASSES))

# Warm-up: train only the new head first, then unfreeze for fine-tuning.
for parameter in model.features.parameters():
    parameter.requires_grad = False

model = model.to(DEVICE)

counts = train_df["label"].value_counts().to_dict()
class_weights = torch.tensor(
    [len(train_df) / (2 * counts[0]), len(train_df) / (2 * counts[1])],
    dtype=torch.float32,
    device=DEVICE,
)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print("class weights    :", [round(float(w), 3) for w in class_weights])
print("trainable params :", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("total params     :", sum(p.numel() for p in model.parameters()))


In [ ]:
# --- Training loop (early stopping on validation loss) -----------------------
from sklearn.metrics import f1_score, roc_auc_score

WARMUP_EPOCHS = 3
PATIENCE = 6
history = []
best = {"val_loss": float("inf"), "epoch": -1, "state": None}


def run_epoch(loader, training):
    """One pass over `loader`; returns (loss, p(fake) per item, labels)."""
    model.train(training)
    losses, probs, targets = [], [], []
    with torch.set_grad_enabled(training):
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits = model(images)
            loss = criterion(logits, labels)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            losses.append(loss.item() * labels.size(0))
            probs.append(torch.softmax(logits, dim=1)[:, 1].detach().cpu())
            targets.append(labels.detach().cpu())
    probs = torch.cat(probs).numpy()
    targets = torch.cat(targets).numpy()
    return sum(losses) / len(targets), probs, targets


for epoch in range(1, EPOCHS + 1):
    if epoch == WARMUP_EPOCHS + 1:
        for parameter in model.features.parameters():
            parameter.requires_grad = True
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR / 5, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - epoch)
        print(f"--- epoch {epoch}: unfroze backbone, lr={LR / 5} ---")

    train_loss, _, _ = run_epoch(loaders["train"], True)
    val_loss, val_probs, val_targets = run_epoch(loaders["val"], False)
    val_f1 = f1_score(val_targets, (val_probs >= 0.5).astype(int), zero_division=0)
    val_auc = (roc_auc_score(val_targets, val_probs)
               if len(set(val_targets.tolist())) > 1 else float("nan"))
    scheduler.step()

    history.append({
        "epoch": epoch,
        "train_loss": round(float(train_loss), 4),
        "val_loss": round(float(val_loss), 4),
        "val_f1": round(float(val_f1), 4),
        "val_auc": round(float(val_auc), 4),
    })
    print(f"ep {epoch:02d} | train {train_loss:.4f} | val {val_loss:.4f} | "
          f"f1 {val_f1:.3f} | auc {val_auc:.3f}")

    if val_loss < best["val_loss"]:
        best = {
            "val_loss": float(val_loss),
            "epoch": epoch,
            "state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
        }
    elif epoch - best["epoch"] >= PATIENCE:
        print(f"early stop at epoch {epoch} (best epoch {best['epoch']})")
        break

model.load_state_dict(best["state"])
print("restored best epoch:", best["epoch"], "| val_loss:", round(best["val_loss"], 4))


In [ ]:
# --- Test-set evaluation -----------------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             precision_recall_fscore_support, roc_auc_score, roc_curve)

_, val_probs, val_targets = run_epoch(loaders["val"], False)
_, test_probs, test_targets = run_epoch(loaders["test"], False)
test_pred = (test_probs >= 0.5).astype(int)

precision, recall, f1, _ = precision_recall_fscore_support(
    test_targets, test_pred, average="binary", zero_division=0
)
metrics = {
    "dataset": KAGGLE_SLUG,
    "n_train": int(len(train_df)),
    "n_val": int(len(val_df)),
    "n_test": int(len(test_df)),
    "accuracy": float(accuracy_score(test_targets, test_pred)),
    "precision_fake": float(precision),
    "recall_fake": float(recall),
    "f1_fake": float(f1),
    "roc_auc": (float(roc_auc_score(test_targets, test_probs))
                if len(set(test_targets.tolist())) > 1 else None),
    "confusion_matrix": confusion_matrix(test_targets, test_pred).tolist(),
    "classification_report": classification_report(
        test_targets, test_pred, target_names=CLASSES, zero_division=0
    ),
    "metadata_only_roc_auc": metadata_auc,
    "pretrained": PRETRAINED,
    "metadata_only_roc_auc_normalised": (float(normalised_auc) if NORMALISE_IMAGES else None),
    "normalise_images": NORMALISE_IMAGES,
    "normalise_max_side": NORMALISE_MAX_SIDE,
    "normalise_orientation": NORMALISE_ORIENTATION,
    "history": history,
    "best_epoch": int(best["epoch"]),
}

print(f"accuracy       : {metrics['accuracy']:.3f}")
print(f"fake precision : {metrics['precision_fake']:.3f}")
print(f"fake recall    : {metrics['recall_fake']:.3f}")
print(f"fake f1        : {metrics['f1_fake']:.3f}")
print(f"roc auc        : {metrics['roc_auc']}")
print(f"metadata-only  : {metadata_auc:.3f}   (shortcut risk)")
print()
print(metrics["classification_report"])

confusion = confusion_matrix(test_targets, test_pred)
fig, ax = plt.subplots(figsize=(4.2, 3.6))
sns.heatmap(confusion, annot=True, fmt="d", cmap="Greens",
            xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
ax.set_xlabel("predicted")
ax.set_ylabel("actual")
ax.set_title("Certificate CNN - test set")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

if metrics["roc_auc"] is not None:
    fpr, tpr, _ = roc_curve(test_targets, test_probs)
    plt.figure(figsize=(4.2, 3.6))
    plt.plot(fpr, tpr, label=f"AUC = {metrics['roc_auc']:.3f}")
    plt.plot([0, 1], [0, 1], "--", color="grey")
    plt.xlabel("false positive rate")
    plt.ylabel("true positive rate")
    plt.legend()
    plt.title("ROC - test set")
    plt.tight_layout()
    plt.savefig("roc_curve.png", dpi=150)
    plt.show()


In [ ]:
# --- Threshold tuning --------------------------------------------------------
# A false positive here means labelling a GENUINE certificate as forged, which is
# worse than missing a fake. We tune on the validation set only and pick a
# threshold that keeps the false-positive rate on genuine certificates at or
# below 5%, never below the default 0.5.
TARGET_FPR = 0.05

genuine_val_probs = val_probs[val_targets == 0]
real_safe_threshold = (float(np.quantile(genuine_val_probs, 1.0 - TARGET_FPR))
                       if len(genuine_val_probs) else 0.5)

best_f1, f1_threshold = -1.0, 0.5
for threshold in np.linspace(0.05, 0.95, 91):
    score = f1_score(val_targets, (val_probs >= threshold).astype(int), zero_division=0)
    if score > best_f1:
        best_f1, f1_threshold = float(score), float(threshold)

CHOSEN_THRESHOLD = float(round(max(real_safe_threshold, 0.5), 2))

print(f"F1-optimal threshold       : {f1_threshold:.2f}  (val f1 {best_f1:.3f})")
print(f"real-safe threshold (FPR<=5%): {real_safe_threshold:.2f}")
print(f"CHOSEN threshold           : {CHOSEN_THRESHOLD:.2f}")

safe_pred = (test_probs >= CHOSEN_THRESHOLD).astype(int)
genuine_mask = test_targets == 0
print(f"  test fake f1             : {f1_score(test_targets, safe_pred, zero_division=0):.3f}")
print(f"  genuine flagged as forged: {int(((safe_pred == 1) & genuine_mask).sum())}"
      f" / {int(genuine_mask.sum())}")

metrics["threshold_f1_optimal"] = f1_threshold
metrics["threshold_real_safe"] = real_safe_threshold
metrics["threshold_used"] = CHOSEN_THRESHOLD


In [ ]:
# --- Export to ONNX + sidecar + metrics --------------------------------------
import onnx
import onnxruntime as ort

ONNX_PATH = "certificate_cnn.onnx"

model.eval()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
torch.onnx.export(
    model,
    dummy,
    ONNX_PATH,
    input_names=["input"],
    output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17,
    do_constant_folding=True,
)
onnx.checker.check_model(onnx.load(ONNX_PATH))

# Cross-check ONNX against PyTorch on a real batch from the test set.
sample_images = next(iter(loaders["test"]))[0].to(DEVICE)
with torch.no_grad():
    torch_probs = torch.softmax(model(sample_images), dim=1).cpu().numpy()

session = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])
raw_logits = session.run(["logits"], {"input": sample_images.cpu().numpy()})[0]
exp = np.exp(raw_logits - raw_logits.max(axis=1, keepdims=True))
onnx_probs = exp / exp.sum(axis=1, keepdims=True)

print("onnxruntime providers :", ort.get_available_providers())
print("max |torch - onnx| p  :", float(np.abs(torch_probs - onnx_probs).max()))

sidecar = {
    "model_file": ONNX_PATH,
    "input_size": IMG_SIZE,
    "mean": MEAN,
    "std": STD,
    "classes": CLASSES,
    "fake_index": 1,
    "threshold": CHOSEN_THRESHOLD,
    "trained_on": KAGGLE_SLUG,
    "pretrained": PRETRAINED,
    "n_train": int(len(train_df)),
    "framework": f"torch {torch.__version__} / onnxruntime {ort.__version__}",
}
with open("certificate_cnn.json", "w") as handle:
    json.dump(sidecar, handle, indent=2)
with open("certificate_metrics.json", "w") as handle:
    json.dump(metrics, handle, indent=2)

print()
print("wrote:", ONNX_PATH, os.path.getsize(ONNX_PATH) // 1024, "KB")
print("wrote: certificate_cnn.json, certificate_metrics.json")
print("wrote: confusion_matrix.png, roc_curve.png")
print()
print(json.dumps(sidecar, indent=2))


## Bring the artifacts into the repo

Download `certificate_cnn.onnx`, `certificate_cnn.json` and `certificate_metrics.json`
plus the two PNGs from the notebook's **Output** panel, then place them:

```
ml/artifacts/certificate_cnn.onnx              # model the backend loads (~2-6 MB)
ml/artifacts/certificate_cnn.json              # input size, mean/std, classes, threshold
ml/evaluation/certificate_metrics.json         # accuracy / F1 / AUC / confusion matrix
ml/evaluation/plots/certificate_confusion_matrix.png
ml/evaluation/plots/certificate_roc_curve.png
```

The backend reads the sidecar JSON for `input_size`, `mean`, `std`, `classes` and
`threshold`, so re-tuning the threshold never requires Python code changes.

## How to read the metrics honestly

| Signal | Meaning |
|---|---|
| `metadata_only_roc_auc` | Raw baseline. **Measured 0.966 on this dataset** — width/height/file size alone nearly separate real from fake. That is a build artifact of the dataset, not forgery. |
| `metadata_only_roc_auc_normalised` | Same baseline after size + orientation normalisation. This is the leakage the CNN can *still* exploit — **quote this number, not the raw one.** |
| `confusion_matrix` | Off-diagonal `real → fake` count is the costliest error: it accuses a genuine certificate. |
| `threshold_real_safe` | Threshold that keeps genuine certificates flagged as forged at 5% or less. |

### Measured shortcut ablation (this dataset, 277 usable images)

| Pipeline | width/height/bytes | bytes only | width/height only |
|---|---|---|---|
| Raw files | 0.966 | 0.969 | 0.966 |
| + size normalised (short side 384) | 0.889 | 0.647 | 0.842 |
| + orientation normalised (portrait) | **0.629** | 0.640 | 0.644 |

Bytes stop being a cue as soon as everything is rescaled (0.969 → 0.647), but
**orientation** remains strong (87% of genuine certificates are portrait vs 19% of the
fakes), so rotating landscape scans upright is what actually removes the shortcut.
The median file size becomes 39 KB (real) vs 35 KB (fake) — nearly identical.

## What to improve next

1. **Judge the normalised baseline, not the raw one.** If
   `metadata_only_roc_auc_normalised` is still above ~0.80, the classes remain
   partly separable without reading content — ship the CNN as a weak signal only.
2. **More data.** 287 images is tiny, and the raw classes differ 35x in size. Add your
   own genuine certificates as `real` and generate fakes in Canva/Photoshop
   (`ml/data/raw/certificates/`).
3. **Label richer forgeries.** Screenshots with mismatched fonts, patched names or
   pasted seals teach real forgery cues; resolution alone does not.
4. **OCR + text forensics.** Feed Tesseract OCR text into the existing
   `backend/app/analyzers/document_analysis.py` rules so certificate *images* also get
   deposit / contact / company checks — not just the CNN verdict.
5. **Multi-page PDFs.** Rasterise each page and score the most suspicious one.
6. **Keep it secondary.** Ship with a small weight; require corroborating evidence
   before it can move the risk band.
